# The Quant Research Workflow

You now know what an edge is and how expectancy decides whether a strategy makes money. But a good idea is worthless without a disciplined process to test it, validate it, and run it safely. This lesson lays out that process end to end: the quant research workflow. It is the backbone of everything you'll do, and the rest of the course is organized around it.

By the end of this lesson you will be able to:

1. Name every stage of the research pipeline and what each one is for
2. Practice hypothesis-first research instead of data mining
3. Keep a research log that protects you from fooling yourself
4. Organize a project so your results are reproducible
5. Explain why skipping any stage quietly kills strategies



## 1. The Pipeline at a glance:

Every serious strategy travels the same road. You met an early version of this map in the first lesson; here we walk it in depth.

```
idea -> data -> signal -> backtest -> validation -> risk sizing -> execution -> monitoring

```
Think of it as a funnel. Many ideas enter the top; very few survive to live trading. The purpose of the pipeline is not to push ideas through — it's to kill *bad ideas as cheaply* and *early as possible*, so you only spend real money on the survivors.

Each stage is more expensive than the last. Rejecting an idea at the hypothesis stage costs a few minutes; at the backtest stage, an afternoon; after deploying real capital, actual losses. A rational researcher front-loads the cheap, brutal filters and treats reaching the live stage as a privilege an idea has to earn.




## Stage 1: Idea(Hypothesis-First)

Research begins with a written hypothesis, not with a dataset and a search for patterns. The difference is everything.

Hypothesis-first means you state, in plain English, **why an edge should exist* before you look at any results.For example: *"Stocks that dropped sharply on no fundamental news tend to bounce within a few days, because forced sellers (margin calls, fund redemptions) temporarily push price below fair value."*. That sentence makes a claim about cause. It tells you what to measure and how to test.

**Data mining is the opposite**: you scan thousands of indicators and parameter combinations and keep whatever looked profitable. With enough combinations, something always looks profitable by pure chance. You will have found noise and named it a strategy.

>If you can't write down why the edge exists before testing, you are not researching — you are pattern-hunting in noise.

A good hypothesis is specific, has an economic rationale, and is *falsifiable* — it makes a prediction that could turn out false. We'll return to falsification shortly.

To make the standard concrete, here is a checklist a hypothesis should pass before you spend any effort testing it:

- **Specific**. It names the instrument, the condition, and the expected behavior. "Stocks bounce" is not specific; "large-cap stocks that fall more than 3 standard deviations in a day on no news tend to recover over the next 3 days" is.
- **Causal**. It says why, naming the participants whose behavior creates the edge. Forced sellers, slow institutions, premium-demanding hedgers — there should be a loser on the other side.
- **Falsifiable**. It predicts something that could be wrong. If no possible result would change your mind, it isn't a hypothesis, it's a belief.
- **Measurable**. Every term maps to something you can compute from data you can actually get.

If a hypothesis fails any of these, fix it before writing a line of code. Most weak strategies can be killed here, for free.

## Stage 2: Data

Once you have a hypothesis, you gather the data needed to test it. This stage is unglamorous and decides more outcomes than any clever model.

Key concerns:
- Point-in-time correctness. The data must reflect what you actually knew at each moment. Using a company's restated earnings, or today's index membership applied to the past, is a form of look-ahead bias.
- Survivorship bias. A dataset that only includes companies that still exist today silently deletes every failure, making any strategy look better than reality.
- Cleanliness. Splits, dividends, bad ticks, and missing days all distort results if unhandled.

Most beginners underestimate this stage. A flawless model on flawed data produces confident nonsense.

A concrete illustration of survivorship bias: suppose you test "buy the 10 largest US stocks each year" using today's list of large companies. By construction that list contains only winners. Firms that were giants in 2005 and later collapsed (Lehman Brothers) are silently missing, so your backtest never experiences their failures. The result looks fabulous and is fictional. The fix is a dataset that records each year's membership as it was known then, failures and all. Whenever a result looks too good, "what survivors am I accidentally including?" should be your first question.

## Stage 3: Signal

Here you translate the hypothesis into a precise calculation — a number or decision the computer produces from the data. The bounce hypothesis above might become: *"signal = 1 when a stock's 3-day return is below the 5th percentile of its own history."*

The discipline at this stage is to keep the signal a direct expression of the hypothesis. Every extra parameter you add (smoothing windows, thresholds, filters) is a knob that can be tuned to fit the past. Fewer knobs means fewer ways to fool yourself.

Here is the bounce signal written out, so you can see how little code a clean hypothesis requires:

In [ ]:
import pandas pd

# returns :daily returns from one stock
three_day = (1+ returns).rolling(3).apply(lambda x:x.prod(), raw =True) - 1

# threshold = the 5th percentile of this stocks own 3-day returns
threshold = three_day.quantile(0.05)


# Signal: go long when the recent 3-days drop is in the worst. 5%
signal = (three_day < threshold).astype(int)

Count the knobs: the 3-day window and the 5th-percentile threshold. That is two parameters, each a direct restatement of the hypothesis ("a sharp recent drop"). The moment you find yourself adding a fourth and fifth knob — a volatility filter here, a volume condition there — pause and ask whether each one expresses your idea or merely improves the backtest. The second kind is how overfitting sneaks in.

## Stage 4: Backtest

A backtest simulates the rule on historical data: it answers "what would have happened if I'd run this in the past?" Done honestly, it's your cheapest test. Done carelessly, it's a generator of false confidence.

An honest backtest:

- Only uses information available at decision time (no look-ahead).
- Includes realistic costs — commissions and slippage — as you learned last lesson.
- Accounts for when you could actually trade (you can't buy at a price that printed before your signal was computable).


The single most common backtest bug is **look-ahead bias** through careless indexing. Consider computing a signal from today's close and then "trading" at today's close — you are using a price you would not have known until the day was over. The fix is to shift your signal forward by one bar so you act on information that was genuinely available:

In [ ]:
#Wrong: Trades on the same bar the signal is computed from
pnl_wrong = signal * returns


#Right :act on yesterdays signal , earning todays return
pnl_right = signal.shift(1) * returns

That single `.shift(1)` is the difference between a fantasy and a test. A backtest with look-ahead routinely shows impossible returns; fix the timing and the magic usually disappears — which is precisely the point.

## Stage 5: Validation

A single good backtest proves almost nothing. The validation stage exists to answer one brutal question: is this result real, or did I overfit?

Core techniques you'll learn later:

- Out-of-sample testing. Develop on one period, then test on data you never looked at.
- Walk-forward analysis. Repeatedly fit on the past and test on the immediate future, mimicking live use.
- Parameter sensitivity. A real edge degrades gracefully as you nudge parameters; an overfit one collapses.
- Multiple-testing awareness. If you tried 500 variants, the best one looks good by luck alone.

Most strategies die here, and that is the system working correctly.

The parameter-sensitivity idea is intuitive once seen. A real edge should not depend on a magic number: if a momentum strategy works with a 60-day lookback, it should also work — perhaps a little less — at 50 or 70 days, sitting on a broad, gentle plateau. An overfit edge is a sharp spike: wonderful at exactly 63 days, terrible at 60 or 66. When you plot performance against a parameter and see a lonely peak, you have found a mirage, not an edge.

## Stage 6: Risk Sizing

A validated edge still must be sized so that an inevitable bad streak doesn't ruin you before the law of large numbers pays off. Position sizing converts a positive expectancy into a survivable equity curve.

This stage decides how much capital each position gets, how much total risk you run, how positions correlate, and what happens in a crash. A profitable signal with reckless sizing is a blow-up waiting to happen. Risk and sizing get their own module.

The core tension: bigger size compounds faster when you're right, but raises the chance a normal losing streak drops you to zero, from which there is no recovery. Sizing is the discipline of staying in the game long enough for the edge to express itself — defense in service of offense.


## Stage 7: Execution

Now you connect to a broker and trade for real. The gap between backtest and reality lives here: orders don't fill instantly, prices move while you act, and your own trading moves the market if you're large. Execution quality directly subtracts from the edge — the slippage term from the expectancy lesson is determined here. Even a great strategy can bleed out through sloppy execution.

## Stage 8: Monitoring

A live strategy is not "done." Markets change, edges decay, and bugs appear. Monitoring means tracking live performance against backtest expectations and knowing, in advance, the signs that the strategy has broken — so you can turn it off before a slow death becomes a disaster.

> Edges decay. The question is never if a strategy stops working, but whether you'll notice in time.


## The Falsification Mindset

The hardest skill in this entire workflow is psychological: you must actively try to disprove your own ideas. Your incentive is to find a strategy that works, which makes you the easiest person in the world to fool. Every flattering result should trigger suspicion, not celebration.

Practically, falsification means asking at each stage: What would I see if this edge were fake? Then you go looking for exactly that. If you can't find it, your confidence is earned. If you stop looking the moment results look good, you've guaranteed eventual disappointment with real money.

This inverts how the mind naturally works. Psychologists call the default **confirmation bias**: we notice evidence supporting what we hope is true and discount the rest. In trading it is expensive — it keeps you running dead strategies and sizing them up right before they fail. The antidote: before each test, write down the specific result that would make you abandon the idea, then run the test honestly and abide by what you wrote.

## Keeping a research log

A research log is a dated, written record of every hypothesis, test, and decision. It is not optional bureaucracy — it is the single most effective defense against self-deception.

A useful log entry records:

- The date and the hypothesis in plain English.
- Exactly what you tested (data range, parameters, costs assumed).
- The result, including failures.
- Your decision and the reasoning behind it.

Why it matters: without a log, you forget you already tried 30 variants, so the 31st "winner" feels like discovery rather than the multiple-testing trap it is. The log also makes results reproducible by your future self.


```
2026-06-28  Hypothesis: oversold large-caps bounce in 3 days (forced selling).
            Data: S&P 500 daily, 2010-2020, survivorship-adjusted.
            Test: long when 3d return < 5th pct; hold 3 days; 10 bps costs.
            Result: expectancy +$11/trade in-sample; -$2 out-of-sample.
            Decision: REJECT. Edge does not survive OOS. Likely overfit threshold.

```

That single honest entry just saved you from trading a dead idea.

The log's quiet superpower is making the multiple-testing problem visible. If you run 30 variants and one looks great, that's roughly what luck alone produces — but only the log remembers the other 29. It's a count of your attempts, not just a diary of wins.

## Organizing a project

Reproducibility starts with structure. A clean, consistent project layout means anyone (including you, in six months) can rerun your work and get the same answer. A simple, effective layout:


```
my-strategy/
    data/          raw and cleaned datasets (never edited by hand)
    notebooks/     exploratory research, dated
    src/           reusable code: signals, backtester, utils
    results/       saved backtest outputs and figures
    research_log.md
    requirements.txt

```

Keep raw data immutable, separate exploration from reusable code, pin your dependencies, and fix random seeds so results don't change run to run. The next lesson sets up exactly this environment in practice.

## Why skipping steps kills strategies
Each stage catches a specific class of failure. Skip it and that failure reaches your live account undetected:

- Skip hypothesis -> you mine noise and call it signal.
- Skip data hygiene -> survivorship and look-ahead inflate everything downstream.
- Skip validation -> overfit results meet real markets and lose.
- Skip risk sizing -> a normal losing streak becomes a blow-up.
- Skip monitoring -> a decayed edge bleeds you out unnoticed.

The pipeline is a series of filters. Every filter you remove lets more bad strategies through to the most expensive possible test: your own money.

## Real-world case study: a strategy that died at every stage
To see the pipeline as a sequence of filters, follow one realistic idea as it gets tested honestly. The hypothesis: "Stocks that close at a new 52-week high keep rising, because breakouts attract momentum buyers." It passes the hypothesis checklist — specific, causal, falsifiable, measurable. Good. It proceeds.

At the data stage, the researcher grabs the current S&P 500 membership and applies it to the last 15 years. A check reveals survivorship bias: many of today's members weren't in the index a decade ago, and failed firms are missing. After switching to a point-in-time membership dataset, the universe is honest. The idea survives, weakened.

At the backtest stage, the first version trades at the same close that prints the new high — look-ahead. After adding .shift(1) to trade on the next day's open, the breakout is often already partly priced in, and returns fall. Still marginally positive. It survives.

At the validation stage, the researcher splits the data: strong in 2010–2017, flat in 2018–2024. A parameter sweep shows the result depends on using exactly a 52-week window — 40 or 60 weeks barely work. That lonely peak screams overfitting, and combined with the weak out-of-sample period the verdict is clear: REJECT. No real money was ever risked. This is not a failure of the researcher; it is the pipeline doing its job, converting a plausible-sounding idea into a cheap "no" instead of an expensive one.

## Common mistakes
- Starting from data instead of a hypothesis. This is data mining in disguise and the most common beginner error.
- Treating the backtest as the finish line. It's a single early filter, not proof.
- Only logging successes. A log of wins teaches you nothing and hides the multiple-testing problem.
- Tuning until it works. Every parameter tweak after seeing results is a step deeper into overfitting.
- Rushing to live trading. Skipping validation or risk sizing to "just try it" is how accounts blow up.